In [58]:
from collections import OrderedDict
from typing import List, Tuple
import copy

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from datasets.utils.logging import disable_progress_bar
from torch.utils.data import DataLoader, TensorDataset

import flwr
from flwr.client import Client, ClientApp, NumPyClient
from flwr.common import Metrics, Context
from flwr.server import ServerApp, ServerConfig, ServerAppComponents
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation
from flwr_datasets import FederatedDataset


import argparse
import pandas as pd
from sklearn.calibration import LabelEncoder
from sklearn.discriminant_analysis import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from imblearn.over_sampling import RandomOverSampler

# DEVICE = torch.device("cpu") 
DEVICE = torch.device("cuda") # Try "cuda" to train on GPU
print(f"Training on {DEVICE}")
print(f"Flower {flwr.__version__} / PyTorch {torch.__version__}")
disable_progress_bar()

Training on cuda
Flower 1.14.0 / PyTorch 2.5.1


In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument('--gpu',
                    type=int,
                    default=0,
                    help="GPU ID, -1 for CPU")
parser.add_argument('--seed',
                    type=int,
                    default=1,
                    help="seed")
parser.add_argument('--repeat', type=int, default=1, help='repeat index')
meta_args = parser.parse_args("")
meta_args.device = torch.device('cuda:{}'.format(meta_args.gpu) if torch.cuda.is_available() and meta_args.gpu != -1 else 'cpu')
meta_args.log_path = "fed_avg"
meta_args.model = "mlp"

# meta_args.model = "cnn"SO FAR GOOD WITHOUT NORMALIZATION 
# meta_args.round = 20
# meta_args.epoch_iterations = 20
# meta_args.local_lr = 0.001
# meta_args.batch_size = 100
# meta_args.decay_weight = 1.0
# meta_args.data_type = ""
meta_args.round = 80 # 50
meta_args.epoch_iterations = 20
meta_args.local_lr = 0.001
meta_args.batch_size = 150
meta_args.decay_weight = 1.0
meta_args.data_type = ""

meta_args.remove_labels = [17, 21, 25, 29]
meta_args.features = ['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max','sender_nic_send_bytes', 'sender_nic_receive_bytes',
            'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes']  
meta_args.filenames = { 
    "wisconsin_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv",
    "wisconsin_hdd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv",
    "wisconsin_hdd_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-hdd-ssd_merged_V3.csv",
    "wisconsin_ssd_delay_10ms_merged":"./ds/v3/selected_cols_merged/wisconsin-220g2-ssd-delayed-10ms_merged_V3.csv",
    }

print(meta_args)

Namespace(gpu=0, seed=1, repeat=1, device=device(type='cuda', index=0), log_path='fed_avg', model='mlp', round=80, epoch_iterations=20, local_lr=0.001, batch_size=150, decay_weight=1.0, data_type='', remove_labels=[17, 21, 25, 29], features=['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max', 'sender_nic_send_bytes', 'sender_nic_receive_bytes', 'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes'], filenames={'wisconsin_ssd_merged': './ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv', 'wisconsin_hdd_merged': './ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv', 'wisconsin_hdd_ssd_merged': './ds/v3/selected_cols_merged/wisconsin-220g2-hdd-ssd_merged_V3.csv', 'wisconsin_ssd_delay_10ms_merged': './ds/v3/selected_cols_merged/wisconsin-220g2-ssd-delayed-10ms_merged_V3.csv'})


In [2]:
class MLPClassifier_torch(nn.Module):
    def __init__(self, input_size, output_size=2, hidden_layer_sizes=(100,),
                 learning_rate=0.001, max_iter=200, tol=1e-4, random_state=None):
        super(MLPClassifier_torch, self).__init__()

        if random_state is not None:
            torch.manual_seed(random_state)

        # Create the network architecture
        layers = []
        prev_size = input_size
        for size in hidden_layer_sizes:
            layers.append(nn.Linear(prev_size, size))
            layers.append(nn.ReLU())
            prev_size = size
        layers.append(nn.Linear(prev_size, output_size))
        # layers.append(nn.Softmax(dim=1))  # Softmax for multi-class classification

        self.model = nn.Sequential(*layers)
        # self.learning_rate = learning_rate
        # self.max_iter = max_iter
        # self.tol = tol
        self.optimizer = None
        # self.criterion = nn.CrossEntropyLoss()  # CrossEntropyLoss for multi-class log loss
        self.criterion = nn.CrossEntropyLoss()  # CrossEntropyLoss for multi-class log loss

    def forward(self, x):
        return self.model(x)

In [42]:
def process_and_prepare_loaders(args, remove_labels=None, features=None, filenames=None):
    if remove_labels is None:
        remove_labels = [17, 21, 25, 29]
    if features is None:
        features = ['sender_avg_rtt_value', 'sender_retrans', 'sender_segs_in', 'sender_tcp_snd_buffer_max','sender_nic_send_bytes', 'sender_nic_receive_bytes',
                    'receiver_seg_out', 'receiver_tcp_rcv_buffer_max', 'receiver_nic_send_bytes', 'receiver_nic_receive_bytes', 'sender_remote_ost_read_bytes', 'receiver_remote_ost_write_bytes']
    if filenames is None:
        filenames = {
            "wisconsin_ssd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_ssd_merged_V3.csv",
            # "wisconsin_ssd_unmerged": "./ds/v3/selected_cols/wisconsin-220g2-10Gbps_ssd_unmerged_V3.csv",

            "wisconsin_hdd_merged": "./ds/v3/selected_cols_merged/wisconsin-220g2-10Gbps_hdd_merged_V3.csv",
            # "wisconsin_hdd_unmerged": "./ds/v3/selected_cols/wisconsin-220g2-10Gbps_hdd_unmerged_V3.csv",
        }
    
    clients_data_loaders = {}
    client_test_loaders = {}
    combined_X_test, combined_y_test = [], []
    test_data_dict = {}

    for client_name, file_path in filenames.items():
        # Step 1: Load the dataset and Label encoding and scaling
        df = pd.read_csv(file_path)
        
        # Step 2: Remove specified labels
        for lbl in remove_labels:
            df = df.drop(df[df.label_value == lbl].index)
        
        # Normalize for transfer learning 
        # df = normalize_df(df)

        X = df.drop(columns="label_value")[features]
        y = df.label_value

        encoder = LabelEncoder()
        scaler = StandardScaler()
        
        y = encoder.fit_transform(y)
        # X = scaler.fit_transform(X)

        # Step 3: Split into train and test sets
        # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        X_train, X_test, y_train, y_test = train_test_split(X,y)
        
       
        # X_train = scaler.fit_transform(X_train)
        # X_test = scaler.transform(X_test)

        # Step 4: Apply oversampling to training data
        X_train, y_train = RandomOverSampler(sampling_strategy="all").fit_resample(X_train, y_train)

        X_train = X_train.to_numpy() if not isinstance(X_train, np.ndarray) else X_train
        X_test = X_test.to_numpy() if not isinstance(X_test, np.ndarray) else X_test
        y_train = y_train.to_numpy() if not isinstance(y_train, np.ndarray) else y_train
        y_test = y_test.to_numpy() if not isinstance(y_test, np.ndarray) else y_test


        # Step 5: Create train DataLoader
        train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                    torch.tensor(y_train, dtype=torch.long))

        ldr_train = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
        # data_loader_list.append(ldr_train)
        clients_data_loaders[client_name] = ldr_train

        # Combine test data for unified test dataset
        combined_X_test.append(X_test)
        combined_y_test.append(y_test)

        # Create individual test DataLoader
        test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                      torch.tensor(y_test, dtype=torch.long))
        
        client_test_loaders[client_name] = DataLoader(test_dataset, batch_size=args.batch_size)

        
    # Combine all test data
    combined_X_test = np.vstack(combined_X_test)
    combined_y_test = np.hstack(combined_y_test)
    total_classes = len(np.unique(combined_y_test))
     # Create combined test DataLoader
    combined_test_dataset = TensorDataset(torch.tensor(combined_X_test, dtype=torch.float32),
                                           torch.tensor(combined_y_test, dtype=torch.long))
    global_test_loader = DataLoader(combined_test_dataset, batch_size=args.batch_size, shuffle=False)

    return clients_data_loaders, client_test_loaders, global_test_loader, total_classes 


In [50]:
def summarize_dataloader(dataloader):
    print("=== DataLoader Summary ===")
    # Dataset length
    dataset_size = len(dataloader.dataset)
    print(f"Total samples: {dataset_size}")
    
    # Batch size
    batch_size = dataloader.batch_size
    print(f"Batch size: {batch_size}")
    
    # Number of batches
    num_batches = len(dataloader)
    print(f"Number of batches: {num_batches}")
    
    # Inspect a single batch
    for i, batch in enumerate(dataloader):
        print(f"Inspecting Batch {i+1}:")
        if isinstance(batch, dict):
            for key, value in batch.items():
                if isinstance(value, (list, tuple)):
                    print(f"  {key}: List/Tuple of length {len(value)}")
                else:
                    print(f"  {key}: Shape {value.shape}, Type {value.dtype}")
        elif isinstance(batch, (list, tuple)):
            for idx, value in enumerate(batch):
                if isinstance(value, torch.Tensor):
                    print(f"  Element {idx}: Shape {value.shape}, Type {value.dtype}")
                else:
                    print(f"  Element {idx}: Type {type(value)}")
        else:
            print("  Batch is not a dict, list, or tuple. Unexpected format.")
        # Only inspect the first batch
        break

    print("===========================")

In [54]:
args = copy.deepcopy(meta_args)
clients_data_loaders, client_test_loaders, global_test_loader, total_classes = process_and_prepare_loaders(args, remove_labels=args.remove_labels, features=args.features, filenames=args.filenames)

print(clients_data_loaders, "\n")
print(client_test_loaders, "\n")
summarize_dataloader(client_test_loaders["wisconsin_ssd_merged"])

{'wisconsin_ssd_merged': <torch.utils.data.dataloader.DataLoader object at 0x7a792f89c980>, 'wisconsin_hdd_merged': <torch.utils.data.dataloader.DataLoader object at 0x7a792fa63bc0>, 'wisconsin_hdd_ssd_merged': <torch.utils.data.dataloader.DataLoader object at 0x7a792f89c0b0>, 'wisconsin_ssd_delay_10ms_merged': <torch.utils.data.dataloader.DataLoader object at 0x7a792f6eec90>} 

{'wisconsin_ssd_merged': <torch.utils.data.dataloader.DataLoader object at 0x7a792f6ed0d0>, 'wisconsin_hdd_merged': <torch.utils.data.dataloader.DataLoader object at 0x7a792f6edca0>, 'wisconsin_hdd_ssd_merged': <torch.utils.data.dataloader.DataLoader object at 0x7a792f89d5b0>, 'wisconsin_ssd_delay_10ms_merged': <torch.utils.data.dataloader.DataLoader object at 0x7a792f6ee8d0>} 

=== DataLoader Summary ===
Total samples: 1408
Batch size: 150
Number of batches: 10
Inspecting Batch 1:
  Element 0: Shape torch.Size([150, 12]), Type torch.float32
  Element 1: Shape torch.Size([150]), Type torch.int64


In [55]:
def load_datasets(partition_id: int):
    client_name = list(args.filenames.keys())[int(partition_id)]
    
    trainloader = clients_data_loaders[client_name]
    testloader = client_test_loaders[client_name]
    valloader = testloader

    return trainloader, valloader, testloader

# load_datasets(2)

In [59]:


def train(net, ldr_train, epochs: int, verbose=False, device=DEVICE, local_lr=0.001):
    loss_func = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=local_lr)
    epochs_losses = []
    net.train()
    for epoch in range(epochs):
        correct, total, epoch_loss = 0, 0, 0.0
        for _, (batch_X, labels) in enumerate(ldr_train):
            batch_X, labels = batch_X.to(device), labels.to(device)
            net.zero_grad()
            # optimizer.zero_grad()
            log_probs = net.forward(batch_X)
            loss = loss_func(log_probs, labels)
            loss.backward()
            optimizer.step()
            # Metrics
            epoch_loss += loss.item()
            total += labels.size(0)
            correct += (torch.max(log_probs.data, 1)[1] == labels).sum().item()
        epoch_loss /= len(ldr_train.dataset)
        epoch_acc = correct / total
        if verbose:
            print(f"Epoch {epoch+1}: train loss {epoch_loss}, accuracy {epoch_acc}")
        epochs_losses.append(epoch_loss)
    w_new = copy.deepcopy(net.state_dict())
    return w_new, sum(epochs_losses) / len(epochs_losses)



def test(net, ldr_test, device=DEVICE):
    net = copy.deepcopy(net).to(device)
    loss_func = nn.CrossEntropyLoss()
    net.eval()
    correct, total, loss = 0, 0, 0.0
    
    all_preds, all_targets = [], []

    with torch.no_grad():
        for index, (data, target) in enumerate(ldr_test):
             data, target = data.to(args.device), target.to(args.device)
             log_probs = net.forward(data)
             test_loss += loss_func(log_probs, target).item()
             _, predicted = torch.max(log_probs, -1) # TODO CHECK FOR GET -1 pr 1 is correct
             
             total += target.size(0)
             correct += predicted.eq(target).sum()
             all_preds.extend(predicted.cpu().numpy())
             all_targets.extend(target.cpu().numpy())
    test_loss /= len(ldr_test.dataset)
    accuracy = 100.00 * correct.item() / total
    f1 = f1_score(all_targets, all_preds, average='weighted')
    return test_loss, accuracy, f1

